# Final Model Validation
## 1. Objective
Test the robustness of the leading log-standard K-Means k=3 and k=4 candidates before making an evidence-based recommendation.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'data/experiments/final_validation'
stability = pd.read_csv(OUT / 'subsampling_stability.csv')
sensitivity = pd.read_csv(OUT / 'feature_sensitivity.csv')
decision = pd.read_csv(OUT / 'final_validation_summary.csv')


## 2. Existing evidence
Earlier experiments favored log-standard preprocessing. K-Means k=3 led full-coverage algorithm comparisons, while k=4 remained an interpretable sensitivity candidate.

## 3. Subsampling methodology
Each candidate is compared with a full-data reference across twenty deterministic 80% samples without replacement. ARI is label invariant. For size and centroid comparisons, sample centroids are optimally matched to reference centroids by minimum total Euclidean distance.

In [ ]:
stability.groupby('k').agg(mean_ARI=('adjusted_rand_index','mean'), std_ARI=('adjusted_rand_index','std'), min_ARI=('adjusted_rand_index','min'), max_ARI=('adjusted_rand_index','max'), mean_size_variation_pp=('mean_absolute_cluster_percentage_difference','mean'), mean_centroid_distance=('mean_centroid_distance','mean'))


## 4. k=3 stability and 5. k=4 stability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
stability.boxplot(column='adjusted_rand_index', by='k', ax=axes[0])
axes[0].set_title('Subsampling ARI'); axes[0].set_xlabel('k'); axes[0].set_ylabel('ARI')
stability.boxplot(column='mean_centroid_distance', by='k', ax=axes[1])
axes[1].set_title('Aligned centroid variation'); axes[1].set_xlabel('k'); axes[1].set_ylabel('Mean Euclidean distance')
fig.suptitle(''); plt.tight_layout(); plt.show()


## 6. MonetaryValue vs TotalItems sensitivity
The alternative replaces MonetaryValue with TotalItems, applies log1p to Frequency, TotalItems, and UniqueProducts, and standardizes all five features.

In [ ]:
sensitivity[['feature_specification','k','silhouette_score','davies_bouldin','calinski_harabasz','smallest_cluster_percentage','largest_cluster_percentage','ari_vs_primary']]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric, title in zip(axes, ['silhouette_score','davies_bouldin','calinski_harabasz'], ['Silhouette (higher)','Davies-Bouldin (lower)','Calinski-Harabasz (higher)']):
    pivot = sensitivity.pivot(index='k', columns='feature_specification', values=metric)
    pivot.plot.bar(ax=ax, title=title); ax.tick_params(axis='x', rotation=0); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 7. Behavioral-profile comparison
Cluster numbers are arbitrary, so interpretation compares median behavior rather than matching numeric labels.

In [ ]:
profile_rows = []
for row in sensitivity.itertuples(index=False):
    profile_rows.extend(json.loads(row.original_unit_cluster_profiles))
profiles = pd.DataFrame(profile_rows)
median_cols = [c for c in profiles if c.startswith('median_')]
values = profiles[median_cols]
normalized = (values - values.mean()) / values.std(ddof=0)
fig, ax = plt.subplots(figsize=(12, 6))
image = ax.imshow(normalized, aspect='auto', cmap='coolwarm', vmin=-2, vmax=2)
ax.set_xticks(range(len(median_cols)), [c.replace('median_','') for c in median_cols], rotation=45, ha='right')
ax.set_yticks(range(len(profiles)), profiles.algorithm + ' k=' + profiles.configuration.str.extract(r'(\d+)')[0] + ' C' + profiles.cluster.astype(str))
ax.set_title('Original-unit cluster medians (column z-scores for display)')
fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()


## 8. Decision framework
Criteria remain separate because internal scores, stability, coverage, interpretability, and parsimony are not commensurable.

In [ ]:
decision.T


## 9. Final recommendation
Recommend K-Means with log-standard preprocessing, the MonetaryValue-based primary feature set, and k=3. It combines the strongest full-coverage internal metrics, balanced and clear profiles, stronger subsampling stability, and parsimony. Keep k=4 as a sensitivity model because its fourth segment is interpretable and its assignments are more robust to replacing MonetaryValue with TotalItems.